# Poke Agent Training

This notebook runs the **temporal transformer** training pipeline via the `poke_agent` package.

- **Edit settings in** `poke_agent/config.py` (epochs, batch size, model size, data paths).
- **Hand tracking:** each game step parses CABT `logs` to infer draw timing, hand age, and opponent hidden-hand signals. Windows and trackers **reset every episode** — no memory across games.
- **Outputs:** checkpoint → `outputs/checkpoints/temporal_current.pt`, report → `outputs/reports/temporal_current.json`.
- **Feature dim:** 283 = 27 coarse (11 base + 16 hand-tracking) + 256 hash. Expect `feature dim 283` after tensor build. Base includes `going_first` from CABT `firstPlayer` (opponent deck rank is **not** a feature).
- **Mac / local:** loads existing rollout JSONL and trains with Torch (CUDA, MPS, or CPU).
- **Linux / Kaggle:** can optionally generate a few CABT rollouts inline when `cg-lib` is available.
- **Does not submit** to the competition leaderboard (use **§10** after training).

**After pulling code changes:** restart the kernel and run all cells from the top.

Docs: `docs/ARCHITECTURE.md`, `docs/poke-agent-modules.md` (see `game_tracker.py`).

## 1. Setup

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

# Ensure repo root is importable before loading poke_agent.
ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from poke_agent.paths import print_runtime_info

print_runtime_info(ROOT)
print("torch", torch.__version__)

repo /home/inzi/poke-bot-agent
python 3.11.15
torch 2.12.1+cu130


## 2. Configuration

Primary settings live in **`poke_agent/config.py`** — edit that file for persistent changes.

This cell builds runtime `CONFIG`. Optional `os.environ[...]` overrides below apply only to this session (see `docs/ARCHITECTURE.md`).

In [2]:
# Training requires CABT evaluation rollouts from the cg.game engine by default.
os.environ.setdefault("REQUIRE_CABT_EVAL_DATA", "1")

# Optional overrides — uncomment to change defaults for this session.
# os.environ["PRIMARY_ROLLOUT_DATA"] = "data/mac-rollouts-100k-fullstate.jsonl"
# os.environ["MODEL_OUTPUT_PATH"] = "outputs/checkpoints/temporal_current.pt"
# os.environ["DATASET_GAMES"] = "5000"   # 5k games to generate and/or cap training
# os.environ["DATASET_GAMES"] = "100000"  # 100k games
# os.environ["BATCH_GAMES"] = "4"  # games per training batch (lower if VRAM is tight)
# os.environ["REQUIRE_CABT_EVAL_DATA"] = "0"  # smoke test only — allows synthetic fallback

from poke_agent.config import build_config
from poke_agent.features import COARSE_BASE_DIM, COARSE_FEATURE_DIM, DERIVED_INFERENCE_DIM
from poke_agent.game_tracker import DERIVED_FEATURE_NAMES

CONFIG = build_config(ROOT)

print("model_id", CONFIG["model_id"])
print("checkpoint", CONFIG["output_path"])
print("report", CONFIG["report_path"])
print("coarse features", COARSE_FEATURE_DIM, f"({COARSE_BASE_DIM} base + {DERIVED_INFERENCE_DIM} hand-tracking)")
print("expected input dim", COARSE_FEATURE_DIM + CONFIG["state_hash_dim"])
print("derived feature names", list(DERIVED_FEATURE_NAMES))
print("output layout")
for name, path in CONFIG["output_layout"].items():
    print(f"  {name}: {path}")

CONFIG

model_id temporal_current
checkpoint /home/inzi/poke-bot-agent/outputs/checkpoints/temporal_current.pt
report /home/inzi/poke-bot-agent/outputs/reports/temporal_current.json
coarse features 29 (13 base + 16 hand-tracking)
expected input dim 285
derived feature names ['self_steps_since_last_draw', 'self_draws_this_turn', 'self_hand_avg_age', 'self_hand_max_age', 'self_cards_played_this_turn', 'self_energy_discarded_since_last_draw', 'self_supporter_cost_before_last_draw', 'opp_steps_since_last_draw', 'opp_draws_this_turn', 'opp_hand_avg_age', 'opp_hand_max_age', 'opp_hand_min_age', 'opp_cards_played_this_turn', 'opp_energy_discarded_since_last_draw', 'opp_hidden_hand_gains', 'opp_visible_deck_to_hand']
output layout
  root: /home/inzi/poke-bot-agent/outputs
  checkpoints: /home/inzi/poke-bot-agent/outputs/checkpoints
  reports: /home/inzi/poke-bot-agent/outputs/reports
  logs: /home/inzi/poke-bot-agent/outputs/logs
  rollouts: /home/inzi/poke-bot-agent/outputs/rollouts
  submissions: /h

{'agent_deck_path': PosixPath('/home/inzi/poke-bot-agent/decks/competitive/high_performing/2026-05_regional-melbourne-2026_10th_mega-lucario.csv'),
 'data_candidates': [PosixPath('/home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/scraped_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/multideck_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/mac-rollouts-10k.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/notebook_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/kaggle-output/data/cabt_rollouts.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/deckpool-smoke.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/container-mp-smoke.jsonl'),
  PosixPath('/home/inzi/poke-bot-agent/data/container-smoke.jsonl')],
 'training_rollout_sources': [PosixPath('/home/inzi/poke-bot-agent/data/scraped_rollouts.jsonl'),


## 3. Device and simulator

In [3]:
from poke_agent.device import torch_device
from poke_agent.simulator import load_simulator, print_simulator_status

DEVICE = torch_device()
print("device", DEVICE)

SIMULATOR = load_simulator(ROOT)
print_simulator_status(SIMULATOR)

device cuda
cg_lib_path /home/inzi/poke-bot-agent/kaggle/input/cg-lib
cg_available True


## 4. Submission deck + optional multi-deck rollout generation

`AGENT_DECK_PATH` / `submission/deck.csv` is the deck we **submit** and pass to beam search at runtime (the model **knows your deck there**, not in training features).

Training must **not** use the same deck for every game. Build merged multi-deck JSONL from scraped replays + weighted archetype matchups.

**Data pipeline (recommended):**
1. `bash scripts/download-episodes-index.sh`
2. `python scripts/scrape_ladder_replays.py --top-percent 1.0`
3. `python scripts/replays_to_rollouts.py --top-percent 1.0 --out data/scraped_rollouts.jsonl`
4. `python scripts/generate_cabt_data.py --episodes 100 --matchups weighted --out data/multideck_rollouts.jsonl`
5. `python scripts/merge_rollouts.py data/scraped_rollouts.jsonl data/multideck_rollouts.jsonl --out data/training_rollouts_merged.jsonl`

Only **complete decisive games** are kept (no truncations/draws/timeouts). Timeout labels use `VALUE_TIMEOUT=-2.0` when present in source data.

Optional inline generation below writes share-weighted archetype matchups to `CONFIG["multideck_rollout_path"]` when the simulator is available.

In [4]:
from poke_agent.config import resolve_generate_games
from poke_agent.data_pipeline import default_cabt_generation_workers, prepare_training_rollout_files
from poke_agent.deck import read_deck

DECK, DECK_SOURCE = read_deck(CONFIG, ROOT)
print("submission deck cards", len(DECK))
print("submission deck source", DECK_SOURCE)
print("merged training data", CONFIG["merged_rollout_path"])
print("training sources", [str(path) for path in CONFIG["training_rollout_sources"]])

GENERATE_GAMES = resolve_generate_games(CONFIG)
WORKERS = default_cabt_generation_workers(episodes=max(GENERATE_GAMES, 1))
print("cabt workers", WORKERS)

if GENERATE_GAMES > 0 and SIMULATOR.available:
    paths = prepare_training_rollout_files(
        CONFIG,
        ROOT,
        simulator_available=True,
        episodes=GENERATE_GAMES,
        workers=WORKERS,
        merge=True,
    )
    print("training will use", paths["training_path"])
else:
    print("skip inline generation; use merged JSONL from scripts/merge_rollouts.py")

submission deck cards 60
submission deck source /home/inzi/poke-bot-agent/decks/competitive/high_performing/2026-05_regional-melbourne-2026_10th_mega-lucario.csv
merged training data /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl
training sources ['/home/inzi/poke-bot-agent/data/scraped_rollouts.jsonl', '/home/inzi/poke-bot-agent/data/multideck_rollouts.jsonl']
cabt workers 30
generate multideck: 2000 games, 30 workers -> /home/inzi/poke-bot-agent/data/multideck_rollouts.jsonl
cg-lib=/home/inzi/poke-bot-agent/kaggle/input/cg-lib
deck0_pool=66
deck1_pool=66
matchups=weighted
games=2000
workers=30
wrote 384804 rows to /home/inzi/poke-bot-agent/data/multideck_rollouts.jsonl
merge rollouts: 1 sources -> /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl
complete games filter: 2000 -> 2000 episodes
merged 1 sources -> 2000 games / 384804 rows -> /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl
training will use /home/inzi/poke-bot-agent/data/training_r

## 5. Load dataset and build tensors

Training prefers `CONFIG["merged_rollout_path"]` (deck-agnostic merged JSONL), then scraped/multi-deck sources, then fallbacks.

`GameEventTracker` walks each game sequentially: one tracker and temporal window per game (no cross-game memory).

When `REQUIRE_COMPLETE_GAMES=1` (default), truncated/draw/timeout episodes are dropped before tensor build. Timeout rows that remain in source data use harsh `VALUE_TIMEOUT=-2.0` labels.

In [5]:
from poke_agent.cabt_validation import (
    assert_cabt_evaluation_rows,
    assert_training_rollout_rows,
    resolve_training_data_path,
    uses_generated_training_data,
)
from poke_agent.dataset import load_jsonl, prepare_training_tensors
from poke_agent.features import COARSE_FEATURE_DIM

EXPECTED_INPUT_DIM = COARSE_FEATURE_DIM + CONFIG["state_hash_dim"]

DATA_PATH = resolve_training_data_path(CONFIG)
if DATA_PATH is None:
    raise RuntimeError(
        "No rollout JSONL found. "
        "Build data/training_rollouts_merged.jsonl via scripts/merge_rollouts.py, "
        "or run section 4 multi-deck generation."
    )

PREVIEW_ROWS = load_jsonl(DATA_PATH)[:5]
if CONFIG.get("require_cabt_eval_data") and not uses_generated_training_data(CONFIG, DATA_PATH):
    assert_cabt_evaluation_rows(PREVIEW_ROWS, path=DATA_PATH, min_rows=1)
else:
    assert_training_rollout_rows(PREVIEW_ROWS, path=DATA_PATH, min_rows=1)
print("using games from", DATA_PATH)

TENSORS = prepare_training_tensors(CONFIG, DEVICE)
print("feature dim", TENSORS.x.shape[1], "(expected", EXPECTED_INPUT_DIM, "with hand tracking)")
print("games", TENSORS.num_games, "steps", TENSORS.x.shape[0], "window", TENSORS.window_size)
if TENSORS.x.shape[1] != EXPECTED_INPUT_DIM:
    raise RuntimeError(
        f"unexpected feature dim {TENSORS.x.shape[1]} (expected {EXPECTED_INPUT_DIM}). "
        "Re-run section 4 to regenerate rollouts with full observations."
    )

jsonl load: 384804 rows using 30 workers
CABT evaluation data OK at /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl: 5 rows, matchup=ogerpon-box vs raging-bolt-ogerpon, feature_dim=29
using games from /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl
jsonl load: 384804 rows using 30 workers
CABT evaluation data OK at /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl: 384804 rows, matchup=ogerpon-box vs raging-bolt-ogerpon, feature_dim=29
training size: 2,000 / 2,000 games (384,804 / 384,804 rows, DATASET_GAMES=2000)
tensor build: 384804 rows across 2000 episodes using 30 workers
loaded 384804 rollout rows from /home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl in 170.6s (30 workers)
training data diversity: 2000 games, 413 matchups, 22 deck slugs
x (384804, 285) value (384804,) transition (384804,) games 2000
history (384804, 512) window 512
feature dim 285 (expected 285 with hand tracking)
games 2000 steps 384804 window 512


## 6. Build model and estimate VRAM

Builds the transformer and prints an estimated GPU memory budget before training starts.

In [6]:
from poke_agent.memory import print_vram_estimate
from poke_agent.training import build_model
from poke_agent.dataset import load_jsonl
from poke_agent.training_diversity import assert_training_pipeline

MODEL = build_model(CONFIG, TENSORS, DEVICE)

TRAIN_ROWS = load_jsonl(DATA_PATH)
assert_training_pipeline(CONFIG, TRAIN_ROWS, TENSORS, MODEL, data_path=DATA_PATH)
print_vram_estimate(
    model=MODEL,
    param_count=sum(p.numel() for p in MODEL.parameters()),
    tensors=TENSORS,
    config=CONFIG,
    device=DEVICE,
)

model: d_model=64 heads=4 layers=4 ff=256 dropout=0.1 window=512
parameters: 286,951
jsonl load: 384804 rows using 30 workers
training diversity OK: 2000 games, 413 matchups, 22 deck slugs. submission deck slug='2026-05_regional-melbourne-2026_10th_mega-lucario'; Model learns from board state; your deck is supplied at runtime (beam/submit).

VRAM estimate for current config
----------------------------------
device: cuda
data: 2,000 games / 384,804 steps x 285 features (window=512, batch_games=2, ~192 steps/game)
parameters: 286,951
dataset tensors: 3.43 GiB
model weights:   1.09 MiB
currently in use: 3.44 GiB allocated, 3.46 GiB reserved
probing one real training step...
measured peak:    7.95 GiB (after 1 forward+backward+adam step)
step overhead:    4.51 GiB above pre-step allocation
gpu present:      11.62 GiB total, 7.40 GiB free right now


## 7. Train

Uses early stopping on total loss. Best weights are restored before checkpoint export.

In [7]:
from poke_agent.training import train_model

TRAINING_REPORT = train_model(MODEL, TENSORS, CONFIG, DEVICE)
TRAINING_REPORT

training batches: games=2000 batch_games=2 batches=1000 (~192 steps/game)
early stop: monitor=value_loss patience=20 min_delta=0.0005 (stop when meaningful improvement plateaus)


training:   0%|          | 0/1000 [00:00<?, ?epoch/s]

epoch 1/1000 batches:   0%|          | 0/1000 [00:00<?, ?batch/s]

epoch 2/1000 batches:   0%|          | 0/1000 [00:00<?, ?batch/s]

epoch 3/1000 batches:   0%|          | 0/1000 [00:00<?, ?batch/s]

...

epoch 44/1000 batches:   0%|          | 0/1000 [00:00<?, ?batch/s]

epoch 45/1000 batches:   0%|          | 0/1000 [00:00<?, ?batch/s]

epoch 46/1000 batches:   0%|          | 0/1000 [00:00<?, ?batch/s]

early stopping at epoch=46; best value_loss=0.00546@26 (no improvement >= 0.0005 for 20 epochs)


{'completed_epochs': 46,
 'requested_epochs': 1000,
 'stopped_early': True,
 'early_stop_metric': 'value_loss',
 'best_total_loss': 11.154180055452674,
 'best_monitor_loss': 0.005459222306913998,
 'best_epoch': 26,
 'last_metrics': {'total_loss': 11.154180055452674,
  'value_loss': 0.005115277812654672,
  'policy_loss': 1.036372310987866,
  'dynamics_loss': 72.60042445740258,
  'entropy': 1.0440742343238514,
  'uncertainty_loss': -4.664442462889717},
 'dataset_rows': 384804,
 'dataset_games': 2000,
 'train_game_limit': 2000,
 'input_dim': 285,
 'window_size': 512,
 'batch_games': 2,
 'device': 'cuda',
 'data_path': '/home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl',
 'reward_scheme': {'value_win': 1.0,
  'value_not_win': -1.0,
  'value_timeout': -2.0},
 'beam_search': {'width': 8, 'time_budget_ms': 1000, 'min_remaining_sec': 120},
 'loss_note': 'Loss is the training objective, not winrate. Winrate requires CABT evaluation games using the model as the action policy.'}

## 8. Save checkpoint and report

Writes `outputs/checkpoints/{model_id}.pt` and JSON training report to `outputs/reports/{model_id}.json`.

In [8]:
from poke_agent.checkpoint import print_training_report, save_checkpoint

OUTPUT_PATH = CONFIG["output_path"]
REPORT_PATH = CONFIG["report_path"]
TRAINING_REPORT = save_checkpoint(
    model=MODEL,
    tensors=TENSORS,
    config=CONFIG,
    training_report=TRAINING_REPORT,
    output_path=OUTPUT_PATH,
)
print_training_report(TRAINING_REPORT, OUTPUT_PATH)
print("report", REPORT_PATH)

saved checkpoint /home/inzi/poke-bot-agent/outputs/checkpoints/temporal_current.pt
saved report /home/inzi/poke-bot-agent/outputs/reports/temporal_current.json

Final training report
----------------------
rows: 384804
window: 512 batch_games: 2
device: cuda
epochs: 46 / 1000
early stopped: True
best total loss: 11.15418 @ epoch 26
total_loss: 11.15418
value_loss: 0.00512
policy_loss: 1.03637
dynamics_loss: 72.60042
entropy: 1.04407
uncertainty_loss: -4.66444

No official Kaggle results found at /home/inzi/poke-bot-agent/data/competition-results.jsonl

Interpretation
- Loss is not winrate.
- Lower value_loss means better win/loss prediction on rollout states.
- Official Kaggle score comes from scripts/fetch_competition_results.py after a submission finishes.
report /home/inzi/poke-bot-agent/outputs/reports/temporal_current.json


## 9. Inspect checkpoint (optional)

## 10. Kaggle submit (before self-play)

Run this after **§8** finishes. Tags a copy as `pre_self_train.pt`, validates the tarball, and uploads to the leaderboard. Then run **§11** self-play.

In [12]:
import shutil
import torch
from poke_agent.features import COARSE_FEATURE_DIM
from poke_agent.kaggle_submit import submit_champion_checkpoint

EXPECTED = COARSE_FEATURE_DIM + CONFIG["state_hash_dim"]
ck = torch.load(CONFIG["output_path"], map_location="cpu", weights_only=False)
if int(ck["input_dim"]) != EXPECTED:
    raise ValueError(
        f"checkpoint input_dim={ck['input_dim']} but features expect {EXPECTED}. "
        "Re-run §5–8 after pulling prize-reward changes before submitting."
    )

PRE_SELF_PLAY_CHECKPOINT = ROOT / "outputs/checkpoints/pre_self_train.pt"
shutil.copy2(CONFIG["output_path"], PRE_SELF_PLAY_CHECKPOINT)

SUBMISSION_MESSAGE = "pre self-play long train (prize rewards + beam)"
SUBMISSION = submit_champion_checkpoint(
    PRE_SELF_PLAY_CHECKPOINT,
    root=ROOT,
    message=SUBMISSION_MESSAGE,
)
SUBMISSION

/home/inzi/miniconda3/envs/poke-bot-agent/lib/python3.11/site-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


submission validation passed: /home/inzi/poke-bot-agent/dist/submission.tar.gz
layout: main.py and deck.csv at archive root (Kaggle requirement)
smoke test: imported main.py and ran agent() on one rollout observation
beam smoke test: choose_action with search_begin_input returned legal indices
built dist/submission.tar.gz with trained model pre_self_train.pt
submitted submission.tar.gz from pre_self_train.pt
Successfully submitted to The Pokémon Company - PTCG AI Battle Challenge Simulation
  0%|          | 0.00/1.51M [00:00<?, ?B/s]
 10%|█         | 160k/1.51M [00:00<00:01, 1.22MB/s]
 50%|████▉     | 768k/1.51M [00:00<00:00, 2.61MB/s]
 69%|██████▉   | 1.05M/1.51M [00:00<00:00, 1.78MB/s]
 89%|████████▊ | 1.34M/1.51M [00:00<00:00, 1.48MB/s]
100%|██████████| 1.51M/1.51M [00:01<00:00, 953kB/s]


{'checkpoint': '/home/inzi/poke-bot-agent/outputs/checkpoints/pre_self_train.pt',
 'tarball': '/home/inzi/poke-bot-agent/dist/submission.tar.gz',
 'message': 'pre self-play long train (prize rewards + beam)',
 'kaggle_output': 'Successfully submitted to The Pokémon Company - PTCG AI Battle Challenge Simulation\n  0%|          | 0.00/1.51M [00:00<?, ?B/s]\n 10%|█         | 160k/1.51M [00:00<00:01, 1.22MB/s]\n 50%|████▉     | 768k/1.51M [00:00<00:00, 2.61MB/s]\n 69%|██████▉   | 1.05M/1.51M [00:00<00:00, 1.78MB/s]\n 89%|████████▊ | 1.34M/1.51M [00:00<00:00, 1.48MB/s]\n100%|██████████| 1.51M/1.51M [00:01<00:00, 953kB/s] \n'}

In [10]:
import torch

checkpoint = torch.load(OUTPUT_PATH, map_location="cpu", weights_only=False)
{
    "model_id": checkpoint.get("model_id"),
    "model_type": checkpoint["model_type"],
    "input_dim": checkpoint["input_dim"],
    "coarse_feature_dim": checkpoint.get("coarse_feature_dim"),
    "policy_dim": checkpoint["policy_dim"],
    "model_config": checkpoint["model_config"],
    "best_total_loss": checkpoint["training_report"]["best_total_loss"],
    "best_epoch": checkpoint["training_report"]["best_epoch"],
    "data_path": checkpoint["data_path"],
    "report_path": checkpoint["training_report"].get("report_path"),
}

{'model_id': 'temporal_current',
 'model_type': 'temporal_transformer_rl_complex_loss',
 'input_dim': 285,
 'coarse_feature_dim': 29,
 'policy_dim': 8,
 'model_config': {'d_model': 64,
  'heads': 4,
  'layers': 4,
  'dim_feedforward': 256,
  'dropout': 0.1,
  'window_size': 512},
 'best_total_loss': 11.154180055452674,
 'best_epoch': 26,
 'data_path': '/home/inzi/poke-bot-agent/data/training_rollouts_merged.jsonl',
 'report_path': '/home/inzi/poke-bot-agent/outputs/reports/temporal_current.json'}

## 11. Curriculum self-play (optional)

**Phase 1 — baseline gate:** fine-tune vs the four **official Kaggle sample agents** (full `main.py` models + `deck.csv`, not deck lists alone):

- Iono's Deck · Dragapult ex · Mega Abomasnow ex · Mega Lucario ex

Install under `baselines/official/` — see `baselines/README.md` and [discussion #708584](https://www.kaggle.com/competitions/pokemon-tcg-ai-battle/discussion/708584).

Each iteration rotates **our** decks from `decks/archetype-samples/` and saves `outputs/checkpoints/by_deck/<slug>.pt` for later comparison.

**Phase 2 — transformer self-play:** starts when aggregate win rate vs all loaded baselines ≥ **60%** (`SELF_PLAY_BASELINE_WIN_RATE`).

| Setting | Meaning |
|---------|---------|
| `SELF_PLAY_GAMES` | CABT games per iteration |
| `SELF_PLAY_BASELINE_WIN_RATE` | Gate to pure transformer self-play (default `0.60`) |
| `SELF_PLAY_AGENT_DECK_DIR` | Our-side deck pool for per-deck fine-tunes |
| `SELF_PLAY_PER_DECK_CHECKPOINT_DIR` | Per-deck model snapshots |

```bash
python scripts/run_self_play.py --curriculum --games 200 --iterations 100
```

In [11]:
from poke_agent.kaggle_submit import DEFAULT_SUBMISSION_MESSAGE
from poke_agent.self_play import run_curriculum_self_play, self_play_settings_from_config

SELF_PLAY_SETTINGS = self_play_settings_from_config(
    CONFIG,
    ROOT,
    agent_name=DECK_SOURCE.stem,
    agent_deck=DECK,
)
# Quick smoke: 1 iteration, 2 games, skip training
# SELF_PLAY_SETTINGS.iterations = 1
# SELF_PLAY_SETTINGS.games_per_iteration = 2
# SELF_PLAY_SETTINGS.train_after_collect = False

SELF_PLAY_REPORTS = run_curriculum_self_play(
    config=CONFIG,
    simulator=SIMULATOR,
    agent_deck=DECK,
    agent_name=DECK_SOURCE.stem,
    settings=SELF_PLAY_SETTINGS,
    device=DEVICE,
    initial_checkpoint=CONFIG["output_path"],
    submit_on_stop=False,
    submission_message=DEFAULT_SUBMISSION_MESSAGE,
    root=ROOT,
)
SELF_PLAY_REPORTS

field deck pool: 159 decks from decks/competitive/high_performing
target eval pool: 159 decks with placement<=1000
self-play device: cuda
self-play iter 1: collect 20 games (workers=20 (inference on cpu), train_device=cuda, current=temporal_current.pt, opponent=temporal_current.pt, beam=True, field=159 decks (sample))


/home/inzi/miniconda3/envs/poke-bot-agent/lib/python3.11/site-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
/home/inzi/miniconda3/envs/poke-bot-agent/lib/python3.11/site-packages/torch/nn/modules/transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torc

KeyboardInterrupt: 